# 07 — GPU calibration and scaled campaign choice

- **Mapped issue:** [#15](https://github.com/majorgilles/transformer-2017-reproduction/issues/15)
- **Depends on:** `06_checkpoint_resume.ipynb` / issue #13.


In [1]:
#| default_exp calibration


## Goal

Before committing to a training run that may last more than a day, this notebook performs a small GPU rehearsal. It measures how long one complete training update takes, how much GPU memory that update occupies, and whether its loss and gradients remain finite. Those measurements let us choose a model and token budget that fit the RTX 4070 SUPER, then estimate the duration of the canonical run in notebook 08.

The benchmark uses synthetic token IDs with realistic tensor shapes. Their numerical token values do not affect the amount of Transformer computation, so the rehearsal can measure model mechanics without reading development or final-test text. Real tokenized WMT data and variable-length collation remain part of the canonical training notebook.


## Paper and project contract

The paper's base model uses six encoder and decoder layers, `d_model=512`, `d_ff=2048`, eight heads, dropout `0.1`, Adam `(0.9, 0.98)` with epsilon `1e-9`, and 4,000 warmup steps. Here, `d_model` is the width of every token representation, `d_ff` is the wider hidden size inside each feed-forward block, and the head count divides attention into parallel representation subspaces. These dimensions strongly influence parameter count, memory use, and step time.

The single-GPU contract permits scaling model width, token budget, data amount, and duration after measurement. We benchmark widths 256, 384, and 512 instead of assuming that the paper-width model fits. Because the measured `d_model=512` candidate retains useful memory headroom, the proposed campaign preserves the base architecture and scales only batch and training resources.


In [2]:
#| export
from dataclasses import dataclass


@dataclass(frozen=True)
class CampaignConfig:
    """Frozen dimensions and resource limits for canonical training."""

    vocabulary_size: int
    d_model: int
    num_heads: int
    d_ff: int
    num_layers: int
    max_sequence_length: int
    dropout: float
    token_budget_per_side: int
    training_examples: int
    max_steps: int
    warmup_steps: int
    label_smoothing: float
    sample_every_steps: int
    sample_count: int
    sample_max_new_tokens: int
    runtime_overhead_multiplier: float


CANONICAL_CAMPAIGN = CampaignConfig(
    vocabulary_size=37_000,
    d_model=512,
    num_heads=8,
    d_ff=2_048,
    num_layers=6,
    max_sequence_length=256,
    dropout=0.1,
    token_budget_per_side=4_096,
    training_examples=1_900_000,
    max_steps=400_000,
    warmup_steps=4_000,
    label_smoothing=0.1,
    sample_every_steps=10_000,
    sample_count=3,
    sample_max_new_tokens=64,
    runtime_overhead_multiplier=1.15,
)

### Frozen data and batching rule

A **training pair** is one English source sentence and its German target translation. Notebook 08 will take the 1,900,000 Europarl training pairs whose tokenized source and complete target are each at most 256 positions, without inspecting development or final-test text. This gives the campaign a reproducible amount and excludes unusually long examples that could exceed the calibrated shape.

A token budget limits padded tensor positions rather than sentence count. Examples will be grouped by similar lengths before the existing batcher packs at most 4,096 padded source positions and 4,096 padded target positions per batch. Short sentences therefore allow more examples in one batch, while long sentences allow fewer. This is a fixed selection rule rather than a claim that all 1,908,920 identified Europarl pairs fit the length boundary.


## Measure representative training steps

A **training step** means one complete update: clear old gradients, run the Transformer forward, calculate label-smoothed loss, backpropagate gradients, and let Adam update the parameters. Each candidate uses the canonical 37,000-token vocabulary, six encoder and decoder layers, eight heads, source and target length 128, and 4,096 positions per side (`batch=32`). The benchmark changes only `d_model` and its corresponding `d_ff=4*d_model`, making the rows directly comparable.

The first two warmup updates are not timed. They allocate Adam's moving-average tensors and allow one-time CUDA setup work to finish. The following five updates determine mean step time and peak allocated VRAM. Peak allocated VRAM is the largest amount of tensor memory PyTorch used during measurement; the assertion keeps it below 90% of total GPU memory to leave practical headroom. Full precision is measured because it already fits, so mixed precision is not required for the proposed campaign.

Synthetic length 128 is representative rather than exhaustive. Real batches vary in shape, so notebook 08 should still stop if an actual batch violates the frozen 4,096-position or 256-position sequence limits.


In [3]:
import gc
from statistics import fmean
from time import perf_counter

import torch

from transformer_2017_reproduction.model import Transformer
from transformer_2017_reproduction.optimization import label_smoothed_loss


def benchmark_candidate(
    d_model: int,
    *,
    token_budget_per_side: int = 4_096,
    sequence_length: int = 128,
    warmup_updates: int = 2,
    measured_updates: int = 5,
) -> dict[str, int | float | bool]:
    """Measure a short full-precision CUDA training fixture."""
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required for GPU calibration")

    batch_size = token_budget_per_side // sequence_length
    if batch_size < 1 or batch_size * sequence_length != token_budget_per_side:
        raise ValueError("token budget must be divisible by sequence length")

    torch.manual_seed(0)
    device = torch.device("cuda")
    model = Transformer(
        vocab_size=CANONICAL_CAMPAIGN.vocabulary_size,
        d_model=d_model,
        num_heads=CANONICAL_CAMPAIGN.num_heads,
        d_ff=4 * d_model,
        max_length=CANONICAL_CAMPAIGN.max_sequence_length,
        pad_token_id=0,
        dropout=CANONICAL_CAMPAIGN.dropout,
        num_layers=CANONICAL_CAMPAIGN.num_layers,
    ).to(device)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-4,
        betas=(0.9, 0.98),
        eps=1e-9,
    )
    source_token_ids = torch.randint(
        4,
        CANONICAL_CAMPAIGN.vocabulary_size,
        (batch_size, sequence_length),
        device=device,
    )
    complete_target_ids = torch.randint(
        4,
        CANONICAL_CAMPAIGN.vocabulary_size,
        (batch_size, sequence_length + 1),
        device=device,
    )
    complete_target_ids[:, 0] = 2
    complete_target_ids[:, -1] = 3

    step_times: list[float] = []
    gradients_are_finite = True
    loss = torch.tensor(float("nan"), device=device)

    for update in range(warmup_updates + measured_updates):
        if update == warmup_updates:
            torch.cuda.reset_peak_memory_stats()

        torch.cuda.synchronize()
        started_at = perf_counter()
        optimizer.zero_grad(set_to_none=True)
        logits = model(source_token_ids, complete_target_ids[:, :-1])
        loss = label_smoothed_loss(
            logits,
            complete_target_ids[:, 1:],
            pad_token_id=0,
            smoothing=CANONICAL_CAMPAIGN.label_smoothing,
        )
        loss.backward()

        if update >= warmup_updates:
            gradients_are_finite = gradients_are_finite and all(
                parameter.grad is None or torch.isfinite(parameter.grad).all().item()
                for parameter in model.parameters()
            )

        optimizer.step()
        torch.cuda.synchronize()

        if update >= warmup_updates:
            step_times.append(perf_counter() - started_at)

    result: dict[str, int | float | bool] = {
        "d_model": d_model,
        "d_ff": 4 * d_model,
        "batch_size": batch_size,
        "positions_per_side": batch_size * sequence_length,
        "parameters": sum(parameter.numel() for parameter in model.parameters()),
        "mean_step_seconds": fmean(step_times),
        "peak_vram_mib": torch.cuda.max_memory_allocated() / 1024**2,
        "loss": loss.detach().item(),
        "finite_loss_and_gradients": bool(torch.isfinite(loss).item()) and gradients_are_finite,
    }

    del model, optimizer, source_token_ids, complete_target_ids, logits, loss
    torch.cuda.empty_cache()
    gc.collect()
    return result


candidate_results = [benchmark_candidate(d_model) for d_model in (256, 384, 512)]

print("GPU calibration (full precision, 4,096 positions per side):")
print("d_model | parameters | mean step | peak VRAM | finite")
for result in candidate_results:
    print(
        f"{result['d_model']:>7} | "
        f"{result['parameters'] / 1_000_000:>8.2f}M | "
        f"{result['mean_step_seconds']:>8.3f}s | "
        f"{result['peak_vram_mib']:>8.0f} MiB | "
        f"{result['finite_loss_and_gradients']}"
    )

GPU calibration (full precision, 4,096 positions per side):
d_model | parameters | mean step | peak VRAM | finite
    256 |    20.53M |    0.143s |     3614 MiB | True
    384 |    39.05M |    0.180s |     4230 MiB | True
    512 |    63.08M |    0.228s |     4862 MiB | True


## Selected campaign and time estimate

The selected row keeps the paper's base dimensions because it fits in full precision with memory headroom. Its **estimated wall time** is the approximate real elapsed time from starting the canonical training command until all planned optimizer steps finish. In other words, it estimates the duration of the training campaign—not the time for this calibration notebook and not model inference time.

The estimate is derived directly from the benchmark:

$$
\text{estimated hours}
= \frac{\text{measured seconds per step} \times 400{,}000 \times 1.15}{3{,}600}.
$$

The `1.15` multiplier adds a simple 15% allowance for work omitted from the synthetic step measurement, such as loading and collating real examples, periodic validation, and checkpoint writing. This is a planning estimate rather than a guarantee: real sentence lengths, operating-system activity, thermal behavior, and validation cadence may change the actual duration. Notebook 08 must report actual elapsed time.


In [4]:
selected_result = next(
    result for result in candidate_results if result["d_model"] == CANONICAL_CAMPAIGN.d_model
)
gpu_total_mib = torch.cuda.get_device_properties(0).total_memory / 1024**2
measured_step_seconds = float(selected_result["mean_step_seconds"])

# Estimate real elapsed training time from the measured update rate.
# The 1.15 multiplier allows 15% for data loading, validation, and checkpoints.
estimated_training_hours = (
    measured_step_seconds
    * CANONICAL_CAMPAIGN.max_steps
    * CANONICAL_CAMPAIGN.runtime_overhead_multiplier
    / 3_600
)

# Preserve the paper's feed-forward ratio and valid per-head dimensions.
assert CANONICAL_CAMPAIGN.d_ff == 4 * CANONICAL_CAMPAIGN.d_model
assert CANONICAL_CAMPAIGN.d_model % CANONICAL_CAMPAIGN.num_heads == 0

# Reject a configuration with numerical failure or less than 10% VRAM headroom.
assert selected_result["finite_loss_and_gradients"] is True
assert float(selected_result["peak_vram_mib"]) < 0.9 * gpu_total_mib
assert estimated_training_hours > 0

print("Frozen canonical campaign:")
print(f"  GPU: {torch.cuda.get_device_name(0)} ({gpu_total_mib:,.0f} MiB)")
print(
    f"  model: d_model={CANONICAL_CAMPAIGN.d_model}, heads={CANONICAL_CAMPAIGN.num_heads}, d_ff={CANONICAL_CAMPAIGN.d_ff}, layers={CANONICAL_CAMPAIGN.num_layers}"
)
print(f"  parameters: {int(selected_result['parameters']):,}")
print(f"  token budget: {CANONICAL_CAMPAIGN.token_budget_per_side:,} padded positions per side")
print(f"  data: {CANONICAL_CAMPAIGN.training_examples:,} eligible Europarl training pairs")
print(f"  duration: {CANONICAL_CAMPAIGN.max_steps:,} optimizer steps")
print(
    f"  sample cadence: {CANONICAL_CAMPAIGN.sample_count} fixed development "
    f"translations every {CANONICAL_CAMPAIGN.sample_every_steps:,} steps"
)
print(f"  measured step: {measured_step_seconds:.3f} seconds")
print(
    f"  peak VRAM: {float(selected_result['peak_vram_mib']):,.0f} MiB ({float(selected_result['peak_vram_mib']) / gpu_total_mib:.1%})"
)
print(f"  estimated wall time: {estimated_training_hours:.1f} hours")

Frozen canonical campaign:
  GPU: NVIDIA GeForce RTX 4070 SUPER (12,282 MiB)
  model: d_model=512, heads=8, d_ff=2048, layers=6
  parameters: 63,082,496
  token budget: 4,096 padded positions per side
  data: 1,900,000 eligible Europarl training pairs
  duration: 400,000 optimizer steps
  sample cadence: 3 fixed development translations every 10,000 steps
  measured step: 0.228 seconds
  peak VRAM: 4,862 MiB (39.6%)
  estimated wall time: 29.1 hours
